In [147]:
import numpy as np
from openai import OpenAI
from dotenv import load_dotenv
from pydantic import BaseModel

In [148]:
import os
from enum import Enum

In [149]:
load_dotenv()

True

In [150]:
LLM_API_URL = os.environ["LLM_API_URL"]
LLM_API_TOKEN = os.environ["LLM_API_TOKEN"]
MODEL = "google/gemma-3-1b"

In [151]:
client = OpenAI(
    base_url=LLM_API_URL,
    api_key=LLM_API_TOKEN
)

In [152]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Hi!"}]
)

print(response.choices[0].message.content)

Hi there! How can I help you today? 😊 

Do you have a question, need some information, or just want to chat? Let me know!


In [153]:
VOID        = 0
PLAYER      = 1
ENNEMY      = 2
GOLD        = 3

SYMBOLS = {VOID: "·", PLAYER: "👤", ENNEMY: "👹", GOLD: "💰"}

In [154]:
initial_map = np.array([
    [0, 0, 0, 0, 0, 0, 0],
    [0, 1, 0, 0, 2, 0, 3], # (1, 1) # (1, 4) # (1, 6)
    [0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 3], # (5, 6)
    [0, 0, 0, 0, 0, 0, 0],
])
initial_map

array([[0, 0, 0, 0, 0, 0, 0],
       [0, 1, 0, 0, 2, 0, 3],
       [0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 3],
       [0, 0, 0, 0, 0, 0, 0]])

# # Couche de contrat

In [155]:
class Direction(str, Enum):
    HAUT       = "HAUT"
    BAS        = "BAS"
    GAUCHE     = "GAUCHE"
    DROITE     = "DROITE"


class PlayerDecision(BaseModel):
    analyse: str
    direction: Direction

In [156]:
MOVES = {
    "HAUT":     (-1, 0),
    "BAS":      ( 1,  0),
    "GAUCHE":   ( 0,  -1),
    "DROITE":   ( 0,   1),
}

# # Moteur de perception

In [157]:
def localize(world_map, entity):
    positions = np.argwhere(world_map == entity)
    return positions

In [158]:
def compute_distances(entities_positions, reference_pos):
    if (len(entities_positions) == 0):
        return np.array([])
    
    v = entities_positions - reference_pos
    distances = np.linalg.norm(v, axis=1)
 
    return np.round(distances, 2)

In [159]:
def perception(world_map):
    player_position = localize(world_map, PLAYER)[0]
    golds_positions = localize(world_map, GOLD)
    ennemies_positions = localize(world_map, ENNEMY)

    # 1. Mouvements physiquement possibles (hors murs)
    mouvements_possibles = []
    for direction, (d_row, d_col) in MOVES.items():
        new_pos = (player_position[0] + d_row, player_position[1] + d_col)
        if allowed_move(world_map, new_pos): 
            mouvements_possibles.append(direction)

    # 2. Directions théoriques vers l'or
    directions_vers_or = []
    if len(golds_positions) > 0:
        golds_distances = compute_distances(golds_positions, player_position)
        nearest_gold_idx = np.argmin(golds_distances)
        delta = golds_positions[nearest_gold_idx] - player_position
        
        if delta[0] > 0: directions_vers_or.append("BAS")
        elif delta[0] < 0: directions_vers_or.append("HAUT")
        if delta[1] > 0: directions_vers_or.append("DROITE")
        elif delta[1] < 0: directions_vers_or.append("GAUCHE")

    # 3. Directions mortelles (ennemi à proximité immédiate)
    directions_dangereuses = []
    if len(ennemies_positions) > 0:
        ennemies_distances = compute_distances(ennemies_positions, player_position)
        nearest_enemy_idx = np.argmin(ennemies_distances)
        delta_ennemi = ennemies_positions[nearest_enemy_idx] - player_position
        
        if delta_ennemi[0] == 1 and delta_ennemi[1] == 0: directions_dangereuses.append("BAS")
        elif delta_ennemi[0] == -1 and delta_ennemi[1] == 0: directions_dangereuses.append("HAUT")
        elif delta_ennemi[0] == 0 and delta_ennemi[1] == 1: directions_dangereuses.append("DROITE")
        elif delta_ennemi[0] == 0 and delta_ennemi[1] == -1: directions_dangereuses.append("GAUCHE")

    # 4. LE FILTRE INTELLIGENT 
    # On croise les listes : l'intersection des chemins possibles, vers l'or, et sans danger.
    choix_recommandes = [d for d in directions_vers_or if d in mouvements_possibles and d not in directions_dangereuses]
    
    # Si la route directe est bloquée, on active l'esquive : tout ce qui est possible et non dangereux
    if not choix_recommandes:
        choix_recommandes = [d for d in mouvements_possibles if d not in directions_dangereuses]

    return {
        "choix_recommandes": choix_recommandes
    }

In [160]:
def show_map(world_map):
    for row in world_map:
        print("\t".join(SYMBOLS.get(cell, "?") for cell in row))
    print('-----------------------------------------------------')

# # Moteur de déplacement

In [161]:
def allowed_move(world_map: np.ndarray, pos):
    n_rows, n_cols = world_map.shape
    r, c = pos

    if r < 0 or c < 0 or r >= n_rows or c >= n_cols:
        return False
    
    return world_map[r, c] in [VOID, GOLD]

In [162]:
def move(world_map: np.ndarray, old_pos, new_pos):
    if not allowed_move(world_map, new_pos):
        return old_pos
    
    entity = world_map[old_pos[0], old_pos[1]]
    world_map[old_pos[0], old_pos[1]] = VOID
    world_map[new_pos[0], new_pos[1]] = entity

    return new_pos

# # Moteur de décision

In [166]:
def decide(player_perception) -> PlayerDecision | None:
    # On récupère les choix recommandés et l'historique
    choix_possibles = player_perception['choix_recommandes'].copy()
    historique = player_perception.get('move_history', [])

    # --- FILTRE ANTI-PING-PONG ---
    if len(historique) > 0:
        dernier_mouvement = historique[-1]
        opposes = {"HAUT": "BAS", "BAS": "HAUT", "GAUCHE": "DROITE", "DROITE": "GAUCHE"}
        mouvement_inverse = opposes.get(dernier_mouvement)

        # Si l'IA a un autre choix que de faire demi-tour, on lui interdit le retour en arrière
        if mouvement_inverse in choix_possibles and len(choix_possibles) > 1:
            choix_possibles.remove(mouvement_inverse)
    # -----------------------------

    prompt = f"""
    Tu es l'IA d'un personnage. 
    
    Le système a calculé pour toi les seules directions sûres et optimales :
    {choix_possibles}

    MISSION :
    - analyse : Confirme ton choix.
    - direction : Choisis EXACTEMENT l'une des directions de la liste.
    """

    response = client.beta.chat.completions.parse(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        response_format=PlayerDecision,
        temperature=0.0
    )
    
    decision = response.choices[0].message.parsed
    if decision:
        print(f"\t 🧠 Décision du LLM : {decision.analyse}")

    return decision

# # Game loop (simulation)


In [167]:
def game_loop(world_map: np.ndarray, max_turns = 10):
    world_map = world_map.copy()
    move_history = []
    
    for turn in range(max_turns):
        print(f"\n =================== [Turn {turn + 1}] ===================")
        show_map(world_map)

        player_pos = localize(world_map, PLAYER)[0]
        
        p = perception(world_map)
        p["move_history"] = move_history

        decision: PlayerDecision | None = decide(p)

        if decision is not None:
            print(f"\t → LLM decision: {decision.direction.value}")

            move_history.append(decision.direction.value)

            d_row, d_col = MOVES[decision.direction.value]
            new_pos = (player_pos[0] + d_row, player_pos[1] + d_col)
            new_pos = move(world_map, player_pos, new_pos)

In [168]:
game_loop(world_map=initial_map, max_turns=10)


 =================== [Turn 1] ===================
·	·	·	·	·	·	·
·	👤	·	·	👹	·	💰
·	·	·	·	·	·	·
·	·	·	·	·	·	·
·	·	·	·	·	·	·
·	·	·	·	·	·	💰
·	·	·	·	·	·	·
-----------------------------------------------------
	 🧠 Décision du LLM : Je confirme mon choix.
	 → LLM decision: DROITE

 =================== [Turn 2] ===================
·	·	·	·	·	·	·
·	·	👤	·	👹	·	💰
·	·	·	·	·	·	·
·	·	·	·	·	·	·
·	·	·	·	·	·	·
·	·	·	·	·	·	💰
·	·	·	·	·	·	·
-----------------------------------------------------
	 🧠 Décision du LLM : Je suis prêt à analyser.
	 → LLM decision: DROITE

 =================== [Turn 3] ===================
·	·	·	·	·	·	·
·	·	·	👤	👹	·	💰
·	·	·	·	·	·	·
·	·	·	·	·	·	·
·	·	·	·	·	·	·
·	·	·	·	·	·	💰
·	·	·	·	·	·	·
-----------------------------------------------------
	 🧠 Décision du LLM : Je suis prêt à analyser.
	 → LLM decision: HAUT

 =================== [Turn 4] ===================
·	·	·	👤	·	·	·
·	·	·	·	👹	·	💰
·	·	·	·	·	·	·
·	·	·	·	·	·	·
·	·	·	·	·	·	·
·	·	·	·	·	·	💰
·	·	·	·	·	·	·
------------------------------